# Problem W3D2: AUC is lognormal, so report the geometric mean

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dc00086-glitch/internship-prep/blob/main/problems/week03/w3d2_scipy_geometric_stats.ipynb)

| | |
|---|---|
| Problem ID | `W3D2` |
| Week / day | 3 / 2 |
| Topic | scipy.stats: lognormal data, geometric mean and CV, normality testing |
| Expected time | 20-30 min |

**How to work this notebook**

1. Run the setup cell to build the data.
2. Run the **Toolkit** cell. It demonstrates every move this problem needs, on a tiny made up example, so none of it is the answer. Read its output before you start typing.
3. Fill in every `# YOUR CODE` blank, one part at a time.
4. Run the check cell as often as you like. It tells you which part is wrong and why, without giving you the answer.
5. When you are done, run the last cell and copy the tutor block.

Stuck on something more basic, like what an array even is? The [foundations notebook](https://colab.research.google.com/github/dc00086-glitch/internship-prep/blob/main/reference/foundations.ipynb) covers Python, numpy, pandas and matplotlib from the ground up, and you can keep it open in a second tab.

Nothing here needs installing. Everything used is already in Colab.


## Scenario

Twenty-four subjects took a single oral dose and you have each one's AUC, the
total exposure in mg*h/L. You have been asked for a one line summary of
central tendency and variability to go at the top of the report.

Exposure parameters are almost always lognormal. Clearance and volume vary
multiplicatively between people, not additively: one subject does not have
"20 units more clearance" than another, they have "30 percent more". Divide
through and the ratios are symmetric even though the raw values are not.

That is why regulatory PK summaries report the **geometric** mean and the
**geometric CV**, and why bioequivalence is assessed on log-transformed AUC
and Cmax. Reporting an arithmetic mean and SD for this data is not quite
wrong, but it describes a bell curve that is not the one you have.

## Multiplicative variability, and the statistics that match it

**What lognormal means.** A quantity is lognormal when its **logarithm** is
normal. Equivalently, it is what you get when many small effects multiply
rather than add. Clearance depends on liver blood flow times enzyme content
times free fraction, each varying by some percentage, so the product is
lognormal almost by construction. The result is strictly positive, has a long
right tail, and has a spread that grows with the magnitude.

**The geometric mean** is the mean computed in log space and brought back:

```python
gm = np.exp(np.mean(np.log(x)))
```

It equals the median of a perfect lognormal distribution, it is always less
than or equal to the arithmetic mean, and it is the value a ratio based
comparison is actually about. When a label says exposure increased 1.8 fold,
that is a ratio of geometric means.

**The geometric CV** is the one people get wrong, because it is not the
ordinary CV of anything. It is derived from the SD **of the logs**:

```python
sd_log = np.std(np.log(x), ddof=1)
geo_cv = 100 * np.sqrt(np.exp(sd_log ** 2) - 1)
```

Taking `100 * SD / mean` on the raw values gives you a different number and
does not have the interpretation of a percentage spread on the ratio scale. If
you have ever wondered why a PK table shows a CV of 45 percent that does not
match anything you can compute from the mean and SD, this is why.

**Testing normality, and what the test is worth.** `stats.shapiro` tests the
null hypothesis that a sample came from a normal distribution. A small p value
is evidence against normality. Two cautions that matter more than the test
itself. With a small sample it has almost no power, so a large p means "not
enough data to object", not "normal". With a very large sample it rejects on
trivial, harmless departures. So treat it as one input alongside a look at the
data, not as a gate that decides your analysis.

Here it will do its job: the raw AUCs should fail and the logs should pass,
which is the expected signature of lognormal data and a tidy confirmation of
what the biology already told you.

## Your task

Three parts, worked one at a time.

| Part | What you work out | Variables to fill in |
|---|---|---|
| a | the arithmetic summary, the one people reach for | `arith_mean`, `arith_sd`, `arith_cv` |
| b | the geometric summary, the one that fits | `geo_mean`, `sd_log`, `geo_cv` |
| c | whether the data backs that up | `p_raw`, `p_log`, `use_log` |

Below this you will find the **Setup** cell, then a **Toolkit** cell that
demonstrates every move on a tiny made up example, then **Part a**, **Part b**
and **Part c**, each with its own explanation and a cell to write in.

## Setup

Run this cell first. It builds the data with a fixed random seed, so your numbers match the checker and match mine.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

PROBLEM_ID = "W3D2"
TOPIC = "scipy.stats: lognormal data, geometric mean and CV, normality testing"

rng = np.random.default_rng(3102)

N_SUBJECTS = 24
# True geometric mean 48 mg*h/L, with 45 percent geometric CV.
_sd_log_true = np.sqrt(np.log(1.0 + 0.45 ** 2))
auc = np.round(rng.lognormal(mean=np.log(48.0), sigma=_sd_log_true, size=N_SUBJECTS), 2)

pk = pd.DataFrame({
    "SUBJECT": ["B%02d" % i for i in range(1, N_SUBJECTS + 1)],
    "AUC": auc,
})

print("pk:", pk.shape)
print(pk.to_string(index=False))
print("")
print("sorted AUC:", np.sort(auc))
print("")
print("notice the long right tail: a few subjects sit far above the rest")


## Toolkit for this problem

| What you want to do | How |
|---|---|
| natural log of an array | `np.log(x)`, and back with `np.exp(x)` |
| arithmetic mean and sample SD | `np.mean(x)`, `np.std(x, ddof=1)` |
| ordinary CV as a percent | `100 * np.std(x, ddof=1) / np.mean(x)` |
| geometric mean | `np.exp(np.mean(np.log(x)))` |
| SD of the logs | `np.std(np.log(x), ddof=1)` |
| geometric CV as a percent | `100 * np.sqrt(np.exp(sd_log ** 2) - 1)` |
| test for normality | `stats.shapiro(x)`, then `.pvalue` |

The geometric CV comes from the SD **of the logs**, not from the raw SD. There
is no way to get it from the arithmetic mean and SD alone, which is why the
two summaries have to be computed separately.

Run the cell below, then carry on to **Part a**, which is the next section down.

In [ ]:
import numpy as np
from scipy import stats

# A tiny made up right skewed sample. Nothing here is the answer.
demo = np.array([10.0, 12.0, 14.0, 15.0, 18.0, 22.0, 30.0, 61.0])
print("demo       :", demo)
print("sorted, the tail is on the right")

print("")
print("--- 1. arithmetic summary ---")
demo_m = float(np.mean(demo))
demo_sd = float(np.std(demo, ddof=1))
print("mean       : %.3f" % demo_m)
print("SD         : %.3f" % demo_sd)
print("CV percent : %.2f" % (100.0 * demo_sd / demo_m))
print("median     : %.3f   <- well below the mean, which is what skew does" % float(np.median(demo)))

print("")
print("--- 2. working in log space ---")
demo_log = np.log(demo)
print("np.log(demo):", np.round(demo_log, 3))
print("mean of logs: %.4f" % float(np.mean(demo_log)))
print("exp of that : %.3f   <- the geometric mean" % float(np.exp(np.mean(demo_log))))
print("")
print("geometric mean is always <= arithmetic mean:")
print("  %.3f <= %.3f" % (float(np.exp(np.mean(demo_log))), demo_m))

print("")
print("--- 3. geometric CV comes from the SD of the LOGS ---")
demo_sdlog = float(np.std(demo_log, ddof=1))
demo_gcv = 100.0 * np.sqrt(np.exp(demo_sdlog ** 2) - 1.0)
print("SD of logs     : %.4f" % demo_sdlog)
print("geometric CV   : %.2f percent" % demo_gcv)
print("ordinary CV    : %.2f percent   <- a different number entirely"
      % (100.0 * demo_sd / demo_m))

print("")
print("--- 4. shapiro tests normality, small p means NOT normal ---")
demo_raw = stats.shapiro(demo)
demo_logged = stats.shapiro(demo_log)
print("raw  : W = %.4f  p = %.4f" % (demo_raw.statistic, demo_raw.pvalue))
print("logs : W = %.4f  p = %.4f" % (demo_logged.statistic, demo_logged.pvalue))
print("")
print("with only 8 values this test has very little power, so do not read too")
print("much into either number. It is one input, not a decision procedure.")


## Part a. The arithmetic summary

Start with the summary most people would write without thinking, so you have
something to compare against.

Compute the arithmetic mean of the AUC values, the sample SD with `n - 1`, and
the ordinary coefficient of variation as a **percent**, which is
`100 * SD / mean`.

Keep these as plain Python floats rather than numpy scalars, which is what
`float(...)` around the result does. It matters less than it used to, but it
keeps the printed output clean.

Compare the mean you get against the median of the same data. The gap between
them is the skew, and it is the reason the rest of this problem exists.

Fill in `arith_mean`, `arith_sd`, `arith_cv`, all plain floats.

Moves: `np.mean`, `np.std(..., ddof=1)`, arithmetic, `float(...)`.

In [ ]:
# Part a. The arithmetic summary.

arith_mean = None   # YOUR CODE: arithmetic mean of auc
arith_sd = None     # YOUR CODE: sample SD, so ddof=1
arith_cv = None     # YOUR CODE: 100 * SD / mean

print("arithmetic mean : %s" % arith_mean)
print("arithmetic SD   : %s" % arith_sd)
print("CV percent      : %s" % arith_cv)
print("median for comparison: %.3f" % float(np.median(auc)))


## Part b. The geometric summary

Now the summary that matches the shape of the data.

The geometric mean is the arithmetic mean computed on the logs and then
exponentiated back. The SD of the logs is a plain sample SD, computed on
`np.log(auc)` rather than on `auc`. The geometric CV as a percent is
`100 * sqrt(exp(sd_log**2) - 1)`.

Two sanity checks once you have them. The geometric mean must come out at or
below the arithmetic mean from part a, always, for any data with any spread in
it. And it should land close to the median, because for lognormal data those
two coincide.

Note that the geometric CV cannot be obtained from `arith_mean` and
`arith_sd`. It needs the SD of the logs, which is why this part computes
`sd_log` explicitly rather than reusing anything from part a.

Fill in `geo_mean`, `sd_log`, `geo_cv`, all plain floats.

Moves: `np.log`, `np.exp`, `np.mean`, `np.std(..., ddof=1)`, `np.sqrt`.

In [ ]:
# Part b. The geometric summary.

geo_mean = None   # YOUR CODE: exp of the mean of the logs
sd_log = None     # YOUR CODE: sample SD of the logs
geo_cv = None     # YOUR CODE: 100 * sqrt(exp(sd_log ** 2) - 1)

print("geometric mean : %s" % geo_mean)
print("SD of logs     : %s" % sd_log)
print("geometric CV   : %s" % geo_cv)
if geo_mean is not None and arith_mean is not None:
    print("")
    print("geometric %.3f <= arithmetic %.3f ?" % (geo_mean, arith_mean),
          geo_mean <= arith_mean)


## Part c. Does the data actually back this up

You have asserted that AUC is lognormal because the biology says so. Check it
against the data rather than leaving it as an article of faith.

Run `stats.shapiro` twice: once on the raw AUC values and once on their logs.
Keep the p value from each. Remember the direction: a **small** p is evidence
**against** normality, so you expect the raw values to produce a small p and
the logs a large one.

Then make the decision explicit. Set `use_log` to `True` if the logs look more
normal than the raw values do, judged by the p values being on opposite sides
of 0.05, and `False` otherwise. Writing the rule down as code, before you see
whether you like the answer, is the discipline this part is really teaching.

One caveat worth carrying forward. Twenty-four values is not many. Had this
test come back ambiguous, the right response would be to fall back on what is
known about exposure parameters rather than to let an underpowered test
overrule the pharmacology.

Fill in `p_raw`, `p_log`, both plain floats, and `use_log`, a single True or
False.

Moves: `stats.shapiro(x)`, `.pvalue`, `np.log`, a comparison.

In [ ]:
# Part c. Test normality on the raw values and on the logs.

p_raw = None    # YOUR CODE: shapiro p value for the raw AUC
p_log = None    # YOUR CODE: shapiro p value for the logged AUC
use_log = None  # YOUR CODE: True if the raw fails and the logs pass at 0.05

print("shapiro p, raw  : %s" % p_raw)
print("shapiro p, logs : %s" % p_log)
print("work on the log scale?", use_log)


## Check your work

Run this cell. It prints PASS, FAIL or TODO for each part. Do not edit it.

In [ ]:
# Grading helpers. You do not need to read or edit this cell.
import base64 as _b64, json as _json, datetime as _dt, textwrap as _tw
import numpy as _np
import pandas as _pd

# Expected answers, encoded so that you do not read them by accident.
_EXP = _json.loads(_b64.b64decode("eyJhcml0aF9jdiI6IDU2LjE4MDg2NDc4OTEwMjA4LCAiYXJpdGhfY3ZfZnJhYyI6IDAuNTYxODA4NjQ3ODkxMDIwOCwgImFyaXRoX21lYW4iOiA1Ny4wODQ1ODMzMzMzMzMzMzUsICJhcml0aF9zZCI6IDMyLjA3MDYxMjU3NzkyMjMwNCwgImFyaXRoX3NkX3BvcCI6IDMxLjM5NTM2NjE2Mjc0ODUzNCwgImdlb19jdiI6IDQ5LjI3MzE0NjIwMzc2ODIsICJnZW9fY3ZfbmFpdmUiOiA0Ni42MjEyNjc3NDQyNzMzNywgImdlb19tZWFuIjogNTAuOTQ4MDMzNzAxODA2OTE0LCAiZ2VvX21lYW5fbG9nMTAiOiA1MC45NDgwMzM3MDE4MDY5MzUsICJtZWFuX29mX2xvZ3MiOiAzLjkzMDgwNjE2NjIyMTUwMjQsICJtZWRpYW4iOiA0OC4xMjk5OTk5OTk5OTk5OTUsICJuIjogMjQsICJwX2xvZyI6IDAuNDkyNjY1MjM2NDk5NzU1LCAicF9yYXciOiAwLjAwMDE2MDQ0Nzk5NjgzNTMyODMyLCAic2RfbG9nIjogMC40NjYyMTI2Nzc0NDI3MzM3LCAic2RfbG9nX3BvcCI6IDAuNDU2Mzk2NTc0MzU0OTk2MywgInVzZV9sb2ciOiB0cnVlfQ==").decode("utf-8"))

CHECK_RESULTS = []


class _Todo(Exception):
    """Raised when a part has not been attempted yet."""


def _need(name):
    g = globals()
    if name not in g:
        raise _Todo("`" + name + "` is not defined yet. Run the starter cell for this part.")
    val = g[name]
    if val is None:
        raise _Todo("`" + name + "` is still None. Fill in the # YOUR CODE blank.")
    return val


def _as_scalar(val):
    try:
        arr = _np.asarray(val, dtype=float)
    except Exception:
        return None
    if arr.size != 1:
        return None
    return float(arr.reshape(-1)[0])


def _as_array(val):
    try:
        return _np.asarray(val, dtype=float)
    except Exception:
        return None


def _num_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_scalar(a)
    if a is None:
        return False
    return bool(_np.isclose(a, float(b), rtol=rtol, atol=atol, equal_nan=True))


def _arr_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_array(a)
    if a is None:
        return False
    b = _np.asarray(b, dtype=float)
    if a.shape != b.shape:
        return False
    return bool(_np.allclose(a, b, rtol=rtol, atol=atol, equal_nan=True))


def _cols_of(obj):
    try:
        return [str(c) for c in obj.columns]
    except Exception:
        return None


def check():
    """Test every part and print PASS, FAIL or TODO for each one."""
    global CHECK_RESULTS
    names = sorted(n for n in globals() if n.startswith("_part_"))
    CHECK_RESULTS = []
    for name in names:
        label = name[len("_part_"):]
        try:
            problem = globals()[name]()
            status = "FAIL" if problem else "PASS"
            msg = problem or ""
        except _Todo as exc:
            status, msg = "TODO", str(exc)
        except Exception as exc:
            status = "FAIL"
            msg = "could not check this part, it raised %s: %s" % (type(exc).__name__, exc)
        CHECK_RESULTS.append((label, status, msg))
    _print_results()


def _print_results():
    bar = "=" * 70
    print(bar)
    print("Problem %s  |  %s" % (PROBLEM_ID, TOPIC))
    print(bar)
    for label, status, msg in CHECK_RESULTS:
        print("Part %s: %s" % (label, status))
        if msg:
            for line in _tw.wrap(msg, width=64):
                print("    " + line)
    n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
    print("-" * 70)
    print("Result: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    if n_pass == len(CHECK_RESULTS):
        print("All parts pass. Run the last cell to build your tutor block.")
    print(bar)


def send_to_tutor():
    """Build the block you copy to your tutor, along with your code."""
    lines = []
    lines.append("===== SEND TO TUTOR =====")
    lines.append("Problem ID : %s" % PROBLEM_ID)
    lines.append("Topic      : %s" % TOPIC)
    lines.append("Date       : %s" % _dt.date.today().isoformat())
    lines.append("Check results:")
    if not CHECK_RESULTS:
        lines.append("  check() has not been run yet. Run the check cell above first.")
    else:
        for label, status, msg in CHECK_RESULTS:
            lines.append("  Part %s: %s" % (label, status))
            if msg:
                for line in _tw.wrap(msg, width=58):
                    lines.append("      " + line)
        n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
        lines.append("Summary: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    lines.append("=========================")
    lines.append("Paste your code cells below this block.")
    return "\n".join(lines)



def _part_a():
    arith_mean = _need("arith_mean")
    arith_sd = _need("arith_sd")
    arith_cv = _need("arith_cv")

    got = _as_scalar(arith_mean)
    if got is None:
        return "`arith_mean` should be a single number."
    if _num_close(got, _EXP["geo_mean"], rtol=1e-6):
        return "`arith_mean` is the geometric mean. Part a wants the plain arithmetic one."
    if _num_close(got, _EXP["median"], rtol=1e-9):
        return "`arith_mean` is the median, not the mean."
    if not _num_close(got, _EXP["arith_mean"], rtol=1e-5):
        return "`arith_mean` is not np.mean of auc."

    got = _as_scalar(arith_sd)
    if got is None:
        return "`arith_sd` should be a single number."
    if _num_close(got, _EXP["arith_sd_pop"], rtol=1e-6):
        return ("`arith_sd` used n in the denominator. numpy defaults to ddof=0, so pass "
                "ddof=1 for the sample SD.")
    if _num_close(got, _EXP["sd_log"], rtol=1e-6):
        return "`arith_sd` is the SD of the logs. Part a works on the raw values."
    if not _num_close(got, _EXP["arith_sd"], rtol=1e-5):
        return "`arith_sd` is not the sample SD of auc."

    got = _as_scalar(arith_cv)
    if got is None:
        return "`arith_cv` should be a single number."
    if _num_close(got, _EXP["arith_cv_frac"], rtol=1e-6):
        return "`arith_cv` is a fraction. The task asks for a percent, so multiply by 100."
    if _num_close(got, _EXP["geo_cv"], rtol=1e-6):
        return "`arith_cv` is the geometric CV. Part a wants 100 * SD / mean on raw values."
    if not _num_close(got, _EXP["arith_cv"], rtol=1e-5):
        return "`arith_cv` is not 100 * arith_sd / arith_mean."
    return None


def _part_b():
    geo_mean = _need("geo_mean")
    sd_log = _need("sd_log")
    geo_cv = _need("geo_cv")

    got = _as_scalar(geo_mean)
    if got is None:
        return "`geo_mean` should be a single number."
    if _num_close(got, _EXP["mean_of_logs"], rtol=1e-6):
        return ("`geo_mean` is the mean of the logs, still on the log scale. Exponentiate "
                "it with np.exp to bring it back to mg*h/L.")
    if _num_close(got, _EXP["arith_mean"], rtol=1e-6):
        return ("`geo_mean` equals the arithmetic mean, so no log transform happened. It is "
                "exp of the mean of np.log(auc).")
    if not _num_close(got, _EXP["geo_mean"], rtol=1e-5):
        if _num_close(got, _EXP["geo_mean_log10"], rtol=1e-4):
            return ("`geo_mean` used log base 10 somewhere without matching it. np.log and "
                    "np.exp are a matched pair; np.log10 needs 10 ** x to undo it.")
        return "`geo_mean` is not np.exp(np.mean(np.log(auc)))."
    if got > _EXP["arith_mean"] * (1 + 1e-9):
        return ("`geo_mean` came out above the arithmetic mean, which is impossible for any "
                "data with spread in it. Something is wrong with the transform.")

    got = _as_scalar(sd_log)
    if got is None:
        return "`sd_log` should be a single number."
    if _num_close(got, _EXP["sd_log_pop"], rtol=1e-6):
        return "`sd_log` used ddof=0. Pass ddof=1 for the sample SD."
    if _num_close(got, _EXP["arith_sd"], rtol=1e-6):
        return "`sd_log` is the SD of the raw values. Take np.log first."
    if not _num_close(got, _EXP["sd_log"], rtol=1e-5):
        return "`sd_log` is not the sample SD of np.log(auc)."

    got = _as_scalar(geo_cv)
    if got is None:
        return "`geo_cv` should be a single number."
    if _num_close(got, _EXP["geo_cv_naive"], rtol=1e-6):
        return ("`geo_cv` is just 100 times the SD of the logs. The formula needs the "
                "exponential: 100 * sqrt(exp(sd_log ** 2) - 1).")
    if _num_close(got, _EXP["arith_cv"], rtol=1e-6):
        return ("`geo_cv` is the ordinary CV of the raw values. The geometric CV is derived "
                "from the SD of the logs instead.")
    if not _num_close(got, _EXP["geo_cv"], rtol=1e-5):
        return "`geo_cv` is not 100 * sqrt(exp(sd_log ** 2) - 1)."
    return None


def _part_c():
    p_raw = _need("p_raw")
    p_log = _need("p_log")
    use_log = _need("use_log")

    got = _as_scalar(p_raw)
    if got is None:
        return ("`p_raw` should be a single number. stats.shapiro returns a result object, "
                "so take .pvalue from it.")
    if _num_close(got, _EXP["p_log"], rtol=1e-6) and not _num_close(got, _EXP["p_raw"], rtol=1e-6):
        return "`p_raw` was computed on the logs. It should test the raw AUC values."
    if not _num_close(got, _EXP["p_raw"], rtol=1e-4):
        return "`p_raw` is not the shapiro p value for the raw AUC values."

    got = _as_scalar(p_log)
    if got is None:
        return "`p_log` should be a single number."
    if _num_close(got, _EXP["p_raw"], rtol=1e-6) and not _num_close(got, _EXP["p_log"], rtol=1e-6):
        return "`p_log` was computed on the raw values. Take np.log first."
    if not _num_close(got, _EXP["p_log"], rtol=1e-4):
        return "`p_log` is not the shapiro p value for np.log(auc)."

    if not isinstance(use_log, (bool, _np.bool_)):
        val = _as_scalar(use_log)
        if val is None:
            return "`use_log` should be a single True or False."
        use_log = bool(val)
    if bool(use_log) != _EXP["use_log"]:
        if _EXP["use_log"]:
            return ("`use_log` should be True here. The raw p is below 0.05, which is "
                    "evidence against normality, and the logged p is above it. Check you "
                    "have the direction the right way round: small p means NOT normal.")
        return "`use_log` does not follow from the two p values and the 0.05 rule."
    return None


check()


## Send to tutor

In [ ]:
# Run this last. Copy everything it prints, then paste your own code under it.
print(send_to_tutor())
